In [4]:
import numpy as np
import pandas as pd
class LDAClassifier:
    def __init__(self):
        self.classes=None
        self.means=None
        self.covariance=None
        self.covariance_inv=None
        self.priors=None
        
    def fit(self,X,y):
        X=np.asarray(X,dtype=float)
        y=np.asarray(y).ravel()
        
        self.classes=np.unique(y)
        
        N=X.shape[0]
        
        p=X.shape[1]
        
        self.means={}
        self.priors={}
        
        pooled_covariance=np.zeros((p,p))
        
        for class_label in self.classes:
            X_class=X[y==class_label]
            N_k=X_class.shape[0]
            mean_vector=np.mean(X_class,axis=0)
            self.means[class_label]=mean_vector
            self.priors[class_label]=N_k/N
            centered_data=X_class-mean_vector
            
            covariance_k=(np.dot(centered_data.T,centered_data)/
                          (N_k-1))
            pooled_covariance+=((N_k-1)*covariance_k)
            
        self.covariance=(pooled_covariance/(N_k))
            
        self.covariance_inv=np.linalg.inv(self.covariance)
        
        return self
    
    def predict(self,X):
        X=np.asarray(X,dtype=float)
        if X.ndim==1:
            X=X.reshape(1,-1)
        predictions=[]
        for x in X:
            discriminant_scores=[]
            for class_label in self.classes:
                mean_vector=self.means[class_label]
                prior=self.priors[class_label]
                
                term1=np.dot(
                    np.dot(x,self.covariance_inv),mean_vector
                )
                
                term2=-0.5*np.dot(
                    np.dot(
                        mean_vector,self.covariance_inv
                    ),mean_vector
                )
                
                term3=np.log(prior)
                
                score=term1+term2+term3
                discriminant_scores.append(score)
            best_index=np.argmax(discriminant_scores)
            predicted_class=self.classes[best_index]
            predictions.append(predicted_class)
        return np.array(predictions)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix

df=pd.read_csv('iris.csv')
x = df.drop('target', axis=1) 
y = df['target'] 

x = (x - x.mean()) / x.std() 

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0) 

model = LDAClassifier()
model.fit(x_train, y_train) 

y_pred = model.predict(x_test) 
print("Predicted values:",y_pred)
print("\nConfusion Matrix:",confusion_matrix(y_test,y_pred))
print("\nAccuracy Score:", accuracy_score(y_test, y_pred))


Predicted values: [2 1 0 2 0 2 0 1 1 1 2 1 1 1 1 0 1 1 0 0 2 1 0 0 2 0 0 1 1 0 2 1 0 2 2 1 0
 2 1 1 2 0 2 0 0]

Confusion Matrix: [[16  0  0]
 [ 0 17  1]
 [ 0  0 11]]

Accuracy Score: 0.9777777777777777
